## 0. Importar Librerías y Cargar Datos

Importamos las librerías necesarias y cargamos los datasets desde la carpeta `datasets/`.

In [1]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
from pathlib import Path
import json
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
import warnings

# Configuración
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Librerías importadas exitosamente")

✅ Librerías importadas exitosamente


In [2]:
# Definir ruta de la carpeta de datasets
datasets_folder = Path('datasets')

# Verificar que la carpeta exista
if not datasets_folder.exists():
    raise FileNotFoundError(f"La carpeta '{datasets_folder}' no existe. Ejecute primero el notebook eda.ipynb para generar los datasets.")

print("="*80)
print("CARGANDO DATASETS DESDE CARPETA LOCAL")
print("="*80)

# Cargar metadata
metadata_path = datasets_folder / 'metadata.json'
if metadata_path.exists():
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    print(f"\n📋 Metadata cargada: {len(metadata['datasets'])} datasets disponibles")
    print(f"📅 Fecha de extracción: {metadata['fecha_extraccion']}")
else:
    print("\n⚠️ Archivo metadata.json no encontrado")
    metadata = None

# Mapeo de archivos CSV
dataset_files = {
    'delitos_bucaramanga': 'delitos_bucaramanga.csv',
    'info_delictiva_bucaramanga': 'info_delictiva_bucaramanga.csv',
    'delitos_sexuales': 'delitos_sexuales.csv',
    'violencia_intrafamiliar': 'violencia_intrafamiliar.csv',
    'hurto_modalidades': 'hurto_modalidades.csv'
}

# Cargar todos los datasets
dataframes = {}

for key, filename in dataset_files.items():
    filepath = datasets_folder / filename
    
    if filepath.exists():
        print(f"\n📂 Cargando: {filename}")
        df = pd.read_csv(filepath, encoding='utf-8')
        dataframes[key] = df
        print(f"   ✅ Cargado: {len(df):,} registros × {len(df.columns)} columnas")
        print(f"   📊 Tamaño en memoria: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    else:
        print(f"\n⚠️ Archivo no encontrado: {filename}")

print(f"\n{'='*80}")
print(f"✅ CARGA COMPLETADA")
print(f"{'='*80}")
print(f"\n📦 Datasets cargados: {len(dataframes)}")
print(f"📊 Total de registros: {sum(len(df) for df in dataframes.values()):,}")

# Mostrar resumen
print(f"\n📋 RESUMEN DE DATASETS CARGADOS:\n")
summary_data = []
for key, df in dataframes.items():
    summary_data.append({
        'Dataset': key,
        'Registros': f"{len(df):,}",
        'Columnas': len(df.columns),
        'Memoria_MB': f"{df.memory_usage(deep=True).sum() / 1024**2:.2f}"
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

CARGANDO DATASETS DESDE CARPETA LOCAL

📋 Metadata cargada: 5 datasets disponibles
📅 Fecha de extracción: 2025-11-20T14:48:14.065952

📂 Cargando: delitos_bucaramanga.csv
   ✅ Cargado: 135,076 registros × 19 columnas
   ✅ Cargado: 135,076 registros × 19 columnas
   📊 Tamaño en memoria: 135.11 MB

📂 Cargando: info_delictiva_bucaramanga.csv
   📊 Tamaño en memoria: 135.11 MB

📂 Cargando: info_delictiva_bucaramanga.csv
   ✅ Cargado: 120,940 registros × 26 columnas
   ✅ Cargado: 120,940 registros × 26 columnas
   📊 Tamaño en memoria: 143.10 MB

📂 Cargando: delitos_sexuales.csv
   ✅ Cargado: 21,859 registros × 9 columnas
   📊 Tamaño en memoria: 10.31 MB

📂 Cargando: violencia_intrafamiliar.csv
   📊 Tamaño en memoria: 143.10 MB

📂 Cargando: delitos_sexuales.csv
   ✅ Cargado: 21,859 registros × 9 columnas
   📊 Tamaño en memoria: 10.31 MB

📂 Cargando: violencia_intrafamiliar.csv
   ✅ Cargado: 50,864 registros × 8 columnas
   📊 Tamaño en memoria: 18.17 MB

📂 Cargando: hurto_modalidades.csv
   ✅ Ca

,Dataset,Registros,Columnas,Memoria_MB
0,delitos_bucaramanga,"135,076",19,135.11
1,info_delictiva_bucaramanga,"120,940",26,143.10
2,delitos_sexuales,"21,859",9,10.31
3,violencia_intrafamiliar,"50,864",8,18.17
4,hurto_modalidades,"1,445",9,0.62


In [3]:
# Inspeccionar formato de fechas en info_delictiva_bucaramanga
print("="*80)
print("INSPECCIONANDO FORMATO DE FECHAS")
print("="*80)

df_info = dataframes['info_delictiva_bucaramanga']

print(f"\n📅 Columna: fecha_hecho")
print(f"Tipo de dato: {df_info['fecha_hecho'].dtype}")
print(f"\nPrimeras 20 muestras de fecha_hecho:")
print(df_info['fecha_hecho'].head(20).tolist())

print(f"\nÚltimas 20 muestras de fecha_hecho:")
print(df_info['fecha_hecho'].tail(20).tolist())

print(f"\nMuestras aleatorias (50):")
print(df_info['fecha_hecho'].sample(50, random_state=42).tolist())

# Verificar si hay patrones comunes
print(f"\n📊 Análisis de patrones:")
sample = df_info['fecha_hecho'].astype(str).head(1000)

# Contar longitudes
print(f"\nLongitudes de strings:")
print(sample.str.len().value_counts().head(10))

# Ver ejemplos por longitud
for length in sample.str.len().unique()[:5]:
    examples = sample[sample.str.len() == length].head(3).tolist()
    print(f"\nLongitud {length}: {examples}")

INSPECCIONANDO FORMATO DE FECHAS

📅 Columna: fecha_hecho
Tipo de dato: object

Primeras 20 muestras de fecha_hecho:
['2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000', '2016-01-01T00:00:00.000']

Últimas 20 muestras de fecha_hecho:
['2025-04-28T00:00:00.000', '2025-05-04T00:00:00.000', '2025-05-08T00:00:00.000', '2025-05-17T00:00:00.000', '2025-05-19T00:00:00.000', '2025-05-24T00:00:00.000', '2025-05-24T00:00:00.000', '2025-02-19T00:00:00.000', '2025-02-19T00:00:00.000', '2025-02-19T00:00:00.000', '2025-03-22T00:00:00.000', '2025-02

# PIPELINES: FEATURE ENGINEERING COMPLETO

**Objetivo:** Crear datasets procesados con todas las features necesarias para entrenar modelos ML, preservando nombres de columnas y permitiendo joins/merges.

**Flujo de trabajo:**
1. **Limpieza básica** - Duplicados, valores erróneos
2. **Feature Engineering Temporal** - Extracción de año, mes, día, día_semana, fin_de_semana
3. **Feature Engineering Espacial** - Grids geográficos, zonas, normalización de coordenadas
4. **Feature Engineering Categórico** - Encoding inteligente con nombres preservados
5. **Feature Engineering de Agregación** - Conteos por zona/fecha, estadísticas móviles
6. **Exportación** - Guardar DataFrames procesados en formato Parquet (eficiente + preserva tipos)

**Formato de salida:** Parquet (mantiene tipos de datos, nombres de columnas, más eficiente que CSV)

## 1. Limpieza de Datos

Aplicamos las correcciones identificadas en el EDA: eliminación de duplicados, limpieza de coordenadas erróneas y normalización de valores faltantes.

In [4]:
print("="*80)
print("LIMPIEZA DE DATOS - APLICANDO CORRECCIONES DEL EDA")
print("="*80)

# Track cleaning actions
cleaning_log = []

# 1. ELIMINAR DUPLICADOS
print("\n🔧 1. ELIMINACIÓN DE DUPLICADOS")
print("-"*80)

duplicates_removed = {}

for dataset_name, df in dataframes.items():
    initial_count = len(df)
    duplicates_count = df.duplicated().sum()
    
    if duplicates_count > 0:
        # Remove duplicates
        df_cleaned = df.drop_duplicates()
        dataframes[dataset_name] = df_cleaned
        
        removed = initial_count - len(df_cleaned)
        duplicates_removed[dataset_name] = removed
        
        print(f"\n📊 {dataset_name}:")
        print(f"   Registros iniciales: {initial_count:,}")
        print(f"   Duplicados encontrados: {duplicates_count:,} ({duplicates_count/initial_count*100:.2f}%)")
        print(f"   Duplicados eliminados: {removed:,}")
        print(f"   Registros finales: {len(df_cleaned):,}")
        
        cleaning_log.append({
            'dataset': dataset_name,
            'action': 'remove_duplicates',
            'records_affected': removed,
            'percentage': f"{removed/initial_count*100:.2f}%"
        })
    else:
        print(f"\n✅ {dataset_name}: Sin duplicados")

if duplicates_removed:
    total_removed = sum(duplicates_removed.values())
    print(f"\n📌 Total de duplicados eliminados: {total_removed:,}")
else:
    print(f"\n✅ No se encontraron duplicados en ningún dataset")

# 2. LIMPIAR COORDENADAS ERRÓNEAS (solo para delitos_bucaramanga)
print("\n\n🔧 2. LIMPIEZA DE COORDENADAS ERRÓNEAS")
print("-"*80)

if 'delitos_bucaramanga' in dataframes:
    df_bucaramanga = dataframes['delitos_bucaramanga']
    
    # Check for placeholder values
    if 'latitud' in df_bucaramanga.columns and 'longitud' in df_bucaramanga.columns:
        errores_lat = df_bucaramanga['latitud'].astype(str).str.contains('xx.xxxx', case=False, na=False)
        errores_lon = df_bucaramanga['longitud'].astype(str).str.contains('yy.yyyy', case=False, na=False)
        
        count_errores_lat = errores_lat.sum()
        count_errores_lon = errores_lon.sum()
        
        if count_errores_lat > 0 or count_errores_lon > 0:
            print(f"\n⚠️ Coordenadas con placeholders detectadas:")
            print(f"   Latitud 'xx.xxxx': {count_errores_lat:,}")
            print(f"   Longitud 'yy.yyyy': {count_errores_lon:,}")
            
            # Replace with NaN
            df_bucaramanga.loc[errores_lat, 'latitud'] = np.nan
            df_bucaramanga.loc[errores_lon, 'longitud'] = np.nan
            
            dataframes['delitos_bucaramanga'] = df_bucaramanga
            
            print(f"   ✅ Placeholders reemplazados con NaN")
            
            cleaning_log.append({
                'dataset': 'delitos_bucaramanga',
                'action': 'clean_coordinate_placeholders',
                'records_affected': count_errores_lat,
                'percentage': f"{count_errores_lat/len(df_bucaramanga)*100:.2f}%"
            })
        else:
            print(f"\n✅ delitos_bucaramanga: No se encontraron placeholders en coordenadas")
    else:
        print(f"\n⚠️ delitos_bucaramanga: Columnas de coordenadas no encontradas")
else:
    print(f"\n⚠️ Dataset 'delitos_bucaramanga' no encontrado")

# 3. NORMALIZAR VALORES "NO REPORTA" (mantener como categoría explícita)
print("\n\n🔧 3. NORMALIZACIÓN DE VALORES 'NO REPORTA'")
print("-"*80)
print("ℹ️ Manteniendo 'NO REPORTA' como categoría explícita (según decisiones del EDA)")
print("   Estos valores se procesarán correctamente en el One-Hot Encoding\n")

# Count "NO REPORTA" values for reference
no_reporta_summary = {}
for dataset_name, df in dataframes.items():
    text_cols = df.select_dtypes(include=['object']).columns
    count = 0
    for col in text_cols:
        count += df[col].astype(str).str.upper().str.contains('NO REPORTA', regex=False, na=False).sum()
    
    if count > 0:
        no_reporta_summary[dataset_name] = count
        print(f"   {dataset_name}: {count:,} valores 'NO REPORTA'")

if not no_reporta_summary:
    print("   No se encontraron valores 'NO REPORTA'")

# 4. RESUMEN DE LIMPIEZA
print("\n\n" + "="*80)
print("RESUMEN DE LIMPIEZA")
print("="*80)

if cleaning_log:
    cleaning_df = pd.DataFrame(cleaning_log)
    print("\nAcciones aplicadas:")
    display(cleaning_df)
else:
    print("\n✅ No se requirieron acciones de limpieza")

# Print final dataset sizes
print("\n📊 TAMAÑOS FINALES DE DATASETS:")
for dataset_name, df in dataframes.items():
    print(f"   {dataset_name}: {len(df):,} registros × {len(df.columns)} columnas")

print("\n✅ LIMPIEZA COMPLETADA")
print("="*80)

LIMPIEZA DE DATOS - APLICANDO CORRECCIONES DEL EDA

🔧 1. ELIMINACIÓN DE DUPLICADOS
--------------------------------------------------------------------------------

✅ delitos_bucaramanga: Sin duplicados

📊 info_delictiva_bucaramanga:
   Registros iniciales: 120,940
   Duplicados encontrados: 217 (0.18%)
   Duplicados eliminados: 217
   Registros finales: 120,723

📊 delitos_sexuales:
   Registros iniciales: 21,859
   Duplicados encontrados: 1,881 (8.61%)
   Duplicados eliminados: 1,881
   Registros finales: 19,978

✅ violencia_intrafamiliar: Sin duplicados

📊 hurto_modalidades:
   Registros iniciales: 1,445
   Duplicados encontrados: 23 (1.59%)
   Duplicados eliminados: 23
   Registros finales: 1,422

📌 Total de duplicados eliminados: 2,121


🔧 2. LIMPIEZA DE COORDENADAS ERRÓNEAS
--------------------------------------------------------------------------------

📊 info_delictiva_bucaramanga:
   Registros iniciales: 120,940
   Duplicados encontrados: 217 (0.18%)
   Duplicados eliminados: 2

,dataset,action,records_affected,percentage
0,info_delictiva_bucaramanga,remove_duplicates,217,0.18%
1,delitos_sexuales,remove_duplicates,1881,8.61%
2,hurto_modalidades,remove_duplicates,23,1.59%



📊 TAMAÑOS FINALES DE DATASETS:
   delitos_bucaramanga: 135,076 registros × 19 columnas
   info_delictiva_bucaramanga: 120,723 registros × 26 columnas
   delitos_sexuales: 19,978 registros × 9 columnas
   violencia_intrafamiliar: 50,864 registros × 8 columnas
   hurto_modalidades: 1,422 registros × 9 columnas

✅ LIMPIEZA COMPLETADA


## 1.1. Validación de Fechas

Verificamos que las columnas `fecha_hecho` se parseen correctamente en cada dataset.

In [5]:
print("="*80)
print("VALIDACIÓN DE COLUMNAS DE FECHA")
print("="*80)

validation_results = []

for dataset_name, df in dataframes.items():
    if 'fecha_hecho' in df.columns:
        print(f"\n📅 Validando: {dataset_name}")
        
        # Contar NaT/nulos ANTES
        initial_nulls = df['fecha_hecho'].isna().sum()
        initial_total = len(df)
        
        # Aplicar parsing robusto multi-formato
        df_temp = df.copy()
        df_temp['fecha_hecho'] = df_temp['fecha_hecho'].replace(['', 'nan', 'NaN', 'None', 'null'], np.nan)
        
        # Estrategia 1: Intentar con ISO 8601 (YYYY-MM-DDTHH:MM:SS.mmm)
        df_temp['fecha_hecho'] = pd.to_datetime(df_temp['fecha_hecho'], format='ISO8601', errors='coerce')
        
        # Estrategia 2: Para valores que fallan con ISO, intentar formato DD/MM/YYYY
        mask_nat = df_temp['fecha_hecho'].isna()
        if mask_nat.any():
            df_temp.loc[mask_nat, 'fecha_hecho'] = pd.to_datetime(
                dataframes[dataset_name].loc[mask_nat, 'fecha_hecho'],
                format='%d/%m/%Y',
                errors='coerce'
            )
        
        # Estrategia 3: Para valores que aún fallan, intentar con dayfirst=True
        mask_nat = df_temp['fecha_hecho'].isna()
        if mask_nat.any():
            df_temp.loc[mask_nat, 'fecha_hecho'] = pd.to_datetime(
                dataframes[dataset_name].loc[mask_nat, 'fecha_hecho'],
                dayfirst=True,
                errors='coerce'
            )
        
        # Actualizar dataframe
        dataframes[dataset_name]['fecha_hecho'] = df_temp['fecha_hecho']
        
        # Contar NaT/nulos DESPUÉS
        final_nulls = dataframes[dataset_name]['fecha_hecho'].isna().sum()
        parsed_successfully = initial_total - final_nulls
        
        print(f"   Total registros: {initial_total:,}")
        print(f"   Nulos iniciales: {initial_nulls:,}")
        print(f"   Parseadas exitosamente: {parsed_successfully:,} ({parsed_successfully/initial_total*100:.1f}%)")
        print(f"   NaT finales: {final_nulls:,} ({final_nulls/initial_total*100:.1f}%)")
        
        if final_nulls > 0:
            print(f"   ⚠️ {final_nulls:,} registros con fecha inválida")
        else:
            print(f"   ✅ Todas las fechas parseadas correctamente")
        
        validation_results.append({
            'Dataset': dataset_name,
            'Total': f"{initial_total:,}",
            'Parseadas': f"{parsed_successfully:,}",
            'NaT': f"{final_nulls:,}",
            'Éxito_%': f"{parsed_successfully/initial_total*100:.1f}%"
        })
    else:
        print(f"\n⚠️ {dataset_name}: No tiene columna 'fecha_hecho'")

print(f"\n{'='*80}")
print("RESUMEN DE VALIDACIÓN DE FECHAS")
print(f"{'='*80}\n")

if validation_results:
    validation_df = pd.DataFrame(validation_results)
    display(validation_df)
else:
    print("No se encontraron columnas 'fecha_hecho' para validar")

print(f"\n✅ VALIDACIÓN DE FECHAS COMPLETADA")
print("="*80)

VALIDACIÓN DE COLUMNAS DE FECHA

⚠️ delitos_bucaramanga: No tiene columna 'fecha_hecho'

📅 Validando: info_delictiva_bucaramanga
   Total registros: 120,723
   Nulos iniciales: 0
   Parseadas exitosamente: 120,723 (100.0%)
   NaT finales: 0 (0.0%)
   ✅ Todas las fechas parseadas correctamente

📅 Validando: delitos_sexuales
   Total registros: 19,978
   Nulos iniciales: 0
   Parseadas exitosamente: 19,978 (100.0%)
   NaT finales: 0 (0.0%)
   ✅ Todas las fechas parseadas correctamente

📅 Validando: violencia_intrafamiliar
   Total registros: 19,978
   Nulos iniciales: 0
   Parseadas exitosamente: 19,978 (100.0%)
   NaT finales: 0 (0.0%)
   ✅ Todas las fechas parseadas correctamente

📅 Validando: violencia_intrafamiliar
   Total registros: 50,864
   Nulos iniciales: 0
   Parseadas exitosamente: 50,864 (100.0%)
   NaT finales: 0 (0.0%)
   ✅ Todas las fechas parseadas correctamente

📅 Validando: hurto_modalidades
   Total registros: 1,422
   Nulos iniciales: 0
   Parseadas exitosamente: 1,4

,Dataset,Total,Parseadas,NaT,Éxito_%
0,info_delictiva_bucaramanga,"120,723","120,723",0,100.0%
1,delitos_sexuales,"19,978","19,978",0,100.0%
2,violencia_intrafamiliar,"50,864","50,864",0,100.0%
3,hurto_modalidades,"1,422","1,422",0,100.0%



✅ VALIDACIÓN DE FECHAS COMPLETADA


## 2. Feature Engineering - Funciones Auxiliares

Definimos funciones para crear features temporales, espaciales y de agregación.

In [6]:
def create_temporal_features(df, date_col='fecha_hecho', date_format='%d/%m/%Y'):
    """
    Crea features temporales a partir de una columna de fecha.
    
    Features creadas:
    - año, mes, dia
    - dia_semana (0=Lunes, 6=Domingo)
    - fin_de_semana (binario)
    - trimestre
    - semestre
    - dia_mes (día del mes)
    - semana_año (semana del año)
    """
    df = df.copy()
    
    # Convertir a datetime si es string con parsing robusto multi-formato
    if df[date_col].dtype == 'object':
        # Limpiar valores vacíos y placeholders
        df[date_col] = df[date_col].replace(['', 'nan', 'NaN', 'None', 'null'], np.nan)
        
        # Estrategia 1: Intentar con ISO 8601 (YYYY-MM-DDTHH:MM:SS.mmm)
        df[date_col] = pd.to_datetime(df[date_col], format='ISO8601', errors='coerce')
        
        # Estrategia 2: Para valores que fallan con ISO, intentar formato DD/MM/YYYY
        mask_nat = df[date_col].isna()
        if mask_nat.any():
            df.loc[mask_nat, date_col] = pd.to_datetime(
                df.loc[mask_nat, date_col], 
                format=date_format, 
                errors='coerce'
            )
        
        # Estrategia 3: Para valores que aún fallan, intentar con dayfirst=True (inferir formato)
        mask_nat = df[date_col].isna()
        if mask_nat.any():
            df.loc[mask_nat, date_col] = pd.to_datetime(
                df.loc[mask_nat, date_col], 
                dayfirst=True, 
                errors='coerce'
            )
    
    # Features temporales básicas
    df['año'] = df[date_col].dt.year
    df['mes'] = df[date_col].dt.month
    df['dia'] = df[date_col].dt.day
    df['dia_semana'] = df[date_col].dt.dayofweek  # 0=Lunes, 6=Domingo
    df['fin_de_semana'] = (df['dia_semana'] >= 5).astype(int)
    
    # Features temporales avanzadas
    df['trimestre'] = df[date_col].dt.quarter
    df['semestre'] = ((df['mes'] - 1) // 6) + 1
    df['dia_mes'] = df[date_col].dt.day
    df['semana_año'] = df[date_col].dt.isocalendar().week
    
    # Nombres de día y mes (para interpretabilidad)
    dias = ['Lunes', 'Martes', 'Miercoles', 'Jueves', 'Viernes', 'Sabado', 'Domingo']
    meses = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 
             'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
    
    df['nombre_dia'] = df['dia_semana'].map(lambda x: dias[x] if pd.notna(x) else None)
    df['nombre_mes'] = df['mes'].map(lambda x: meses[int(x)-1] if pd.notna(x) else None)
    
    return df


def create_spatial_features(df, lat_col='latitud', lon_col='longitud', grid_size=0.01):
    """
    Crea features espaciales a partir de coordenadas.
    
    Features creadas:
    - lat_grid, lon_grid (discretización en grids)
    - zona_id (identificador único de zona)
    - distancia_centro (distancia al centro de Bucaramanga: 7.119, -73.123)
    """
    df = df.copy()
    
    # Convertir coordenadas a float si es necesario
    if df[lat_col].dtype == 'object':
        df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    if df[lon_col].dtype == 'object':
        df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
    
    # Crear grids geográficos (para clustering espacial)
    df['lat_grid'] = (df[lat_col] / grid_size).round() * grid_size
    df['lon_grid'] = (df[lon_col] / grid_size).round() * grid_size
    
    # Zona ID (combinación de lat_grid y lon_grid)
    df['zona_id'] = df['lat_grid'].astype(str) + '_' + df['lon_grid'].astype(str)
    
    # Distancia al centro de Bucaramanga (aproximada)
    centro_lat, centro_lon = 7.119, -73.123
    df['dist_centro'] = np.sqrt(
        (df[lat_col] - centro_lat)**2 + (df[lon_col] - centro_lon)**2
    )
    
    return df


def create_categorical_encoding(df, cat_cols, method='label', min_freq=0.01):
    """
    Codifica variables categóricas preservando nombres de columnas.
    
    Métodos:
    - 'label': Label encoding (0, 1, 2, ...)
    - 'frequency': Frequency encoding (proporción de cada categoría)
    - 'target': Target encoding (requiere columna objetivo) - NO implementado aquí
    
    Parámetros:
    - min_freq: Frecuencia mínima para mantener categoría (otras → 'OTRAS')
    """
    df = df.copy()
    
    for col in cat_cols:
        if col not in df.columns:
            continue
        
        # Agrupar categorías raras
        value_counts = df[col].value_counts(normalize=True)
        rare_categories = value_counts[value_counts < min_freq].index.tolist()
        
        if rare_categories:
            df[col] = df[col].replace(rare_categories, 'OTRAS')
        
        # Aplicar encoding
        if method == 'label':
            # Label encoding preservando nombre original
            df[f'{col}_encoded'] = df[col].astype('category').cat.codes
        
        elif method == 'frequency':
            # Frequency encoding
            freq_map = df[col].value_counts(normalize=True).to_dict()
            df[f'{col}_freq'] = df[col].map(freq_map)
    
    return df


def create_aggregation_features(df, group_cols, agg_col='cantidad', window_days=7):
    """
    Crea features de agregación (conteos, promedios móviles, etc.).
    
    Features creadas:
    - count_by_[group]: Conteo por grupo
    - mean_[window]d: Media móvil de N días
    - std_[window]d: Desviación estándar móvil de N días
    """
    df = df.copy()
    
    # Conteo por grupos
    for group_col in group_cols:
        if group_col in df.columns:
            group_counts = df.groupby(group_col).size().rename(f'count_by_{group_col}')
            df = df.join(group_counts, on=group_col)
    
    # Si hay columna de fecha, crear rolling features
    if 'fecha_hecho' in df.columns and agg_col in df.columns:
        df = df.sort_values('fecha_hecho')
        
        # Rolling mean y std (requiere índice temporal)
        df[f'rolling_mean_{window_days}d'] = df[agg_col].rolling(
            window=window_days, min_periods=1
        ).mean()
        
        df[f'rolling_std_{window_days}d'] = df[agg_col].rolling(
            window=window_days, min_periods=1
        ).std()
    
    return df


print("✅ Funciones de feature engineering definidas")

✅ Funciones de feature engineering definidas


## 2.5. Transformador de Coordenadas

Definimos el transformador `ObjectToFloatTransformer` para normalizar coordenadas y valores numéricos mal formateados antes de usarlo en el feature engineering.

In [7]:
from sklearn.base import BaseEstimator, TransformerMixin

class ObjectToFloatTransformer(BaseEstimator, TransformerMixin):
    """
    Transforma columnas string a float, manejando diferentes formatos de decimales.
    
    Maneja dos casos:
    1. Separadores de miles con comas (ej: "7,170,557,382" -> 7.170557382)
    2. Separador decimal con coma (ej: "3,14" -> 3.14)
    
    Para coordenadas geográficas, aplica escalado inteligente para asegurar
    rangos válidos de lat/lon para Bucaramanga (lat: 7.0-7.3, lon: -73.5 a -73.0).
    """
    def __init__(self, columnas):
        self.columnas = columnas

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columnas:
            # Convertir a string primero
            X[col] = X[col].astype(str)
            
            # Verificar si hay múltiples comas (separadores de miles)
            sample_values = X[col].head(100)
            has_multiple_commas = sample_values.str.count(',').max() > 1
            
            if has_multiple_commas:
                # Eliminar todas las comas y convertir a float
                X[col] = X[col].str.replace(',', '', regex=False)
                X[col] = pd.to_numeric(X[col], errors='coerce')
                
                # Normalizar coordenadas - escalado inteligente por valor
                def normalize_coordinate(value):
                    if pd.isna(value):
                        return value
                    
                    abs_val = abs(value)
                    
                    # Ya está en rango válido
                    if abs_val <= 180:
                        return value
                    
                    value_str = str(int(abs_val))
                    num_digits = len(value_str)
                    
                    results = []
                    
                    # Probar como latitud (un dígito antes del decimal: 7.XXXXX)
                    if num_digits > 1:
                        lat_candidate = abs_val / (10 ** (num_digits - 1))
                        if 6.5 <= lat_candidate <= 7.5:
                            results.append(('lat', lat_candidate if value >= 0 else -lat_candidate))
                    
                    # Probar como longitud (dos dígitos antes del decimal: 73.XXXXX)
                    if num_digits > 2:
                        lon_candidate = abs_val / (10 ** (num_digits - 2))
                        if 72.5 <= lon_candidate <= 73.5:
                            results.append(('lon', lon_candidate if value >= 0 else -lon_candidate))
                    
                    # Seleccionar la interpretación válida
                    if results:
                        if len(results) > 1 and value < 0:
                            return results[1][1]  # longitud
                        return results[0][1]
                    
                    # Fallback: escalado genérico
                    scale_factor = 10 ** (num_digits - 2)
                    result = value / scale_factor
                    
                    while abs(result) > 180 and scale_factor < 10**12:
                        scale_factor *= 10
                        result = value / scale_factor
                    
                    return result
                
                X[col] = X[col].apply(normalize_coordinate)
                
            else:
                # Una sola coma es separador decimal: "3,14" -> "3.14"
                X[col] = X[col].str.replace(',', '.', regex=False)
                X[col] = pd.to_numeric(X[col], errors='coerce')
        
        return X

print("✅ Transformador ObjectToFloatTransformer definido")

✅ Transformador ObjectToFloatTransformer definido


## 3. Aplicar Feature Engineering a Cada Dataset

Aplicamos transformaciones específicas según las características de cada dataset.

In [8]:
print("="*80)
print("FEATURE ENGINEERING POR DATASET")
print("="*80)

datasets_procesados = {}

# ============================================================================
# 1. DELITOS_BUCARAMANGA
# ============================================================================
print("\n📊 1. Procesando: delitos_bucaramanga")
print("-"*80)

df_delitos_bga = dataframes['delitos_bucaramanga'].copy()

# PASO PREVIO: Normalizar coordenadas con ObjectToFloatTransformer
if 'latitud' in df_delitos_bga.columns and 'longitud' in df_delitos_bga.columns:
    coord_transformer = ObjectToFloatTransformer(['latitud', 'longitud'])
    df_delitos_bga = coord_transformer.fit_transform(df_delitos_bga)
    print("   ✅ Coordenadas normalizadas (ObjectToFloatTransformer)")

# Spatial features (ahora lat/lon están en formato correcto)
if 'latitud' in df_delitos_bga.columns and 'longitud' in df_delitos_bga.columns:
    df_delitos_bga = create_spatial_features(df_delitos_bga)
    print("   ✅ Features espaciales creadas")

# Categorical encoding
cat_cols = ['armas_medios', 'barrios_hecho', 'zona', 'nom_comuna', 'conducta', 
            'clasificaciones_delito', 'curso_de_vida', 'estado_civil_persona', 
            'genero', 'movil_agresor', 'movil_victima', 'dia_semana']
cat_cols = [c for c in cat_cols if c in df_delitos_bga.columns]

df_delitos_bga = create_categorical_encoding(df_delitos_bga, cat_cols, method='label')
print(f"   ✅ {len(cat_cols)} variables categóricas codificadas")

# Aggregation features
if 'zona_id' in df_delitos_bga.columns:
    df_delitos_bga = create_aggregation_features(df_delitos_bga, ['zona_id', 'barrios_hecho'])
    print("   ✅ Features de agregación creadas")

datasets_procesados['delitos_bucaramanga'] = df_delitos_bga
print(f"   📏 Shape final: {df_delitos_bga.shape}")
print(f"   📋 Columnas: {len(df_delitos_bga.columns)}")

# ============================================================================
# 2. INFO_DELICTIVA_BUCARAMANGA
# ============================================================================
print("\n📊 2. Procesando: info_delictiva_bucaramanga")
print("-"*80)

df_info = dataframes['info_delictiva_bucaramanga'].copy()

# PASO PREVIO: Normalizar edad con ObjectToFloatTransformer
if 'edad' in df_info.columns:
    edad_transformer = ObjectToFloatTransformer(['edad'])
    df_info = edad_transformer.fit_transform(df_info)
    print("   ✅ Edad normalizada (ObjectToFloatTransformer)")

# Temporal features (tiene fecha_hecho)
if 'fecha_hecho' in df_info.columns:
    df_info = create_temporal_features(df_info, date_col='fecha_hecho')
    print("   ✅ Features temporales creadas")

# Spatial features (NO tiene latitud/longitud directas, pero tiene barrios)
# Agregamos conteo por barrio como proxy espacial
if 'barrios_hecho' in df_info.columns:
    barrio_counts = df_info.groupby('barrios_hecho').size().rename('barrio_delitos_count')
    df_info = df_info.join(barrio_counts, on='barrios_hecho')
    print("   ✅ Features espaciales (barrios) creadas")

# Categorical encoding
cat_cols = ['descripcion_conducta', 'armas_medios', 'barrios_hecho', 'sexo', 
            'movil_victima', 'movil_agresor', 'clase_sitio', 'delito_solo', 
            'curso_vida', 'rango_horario', 'tipolog_a', 'dia_nombre', 
            'localidad', 'nom_com']
cat_cols = [c for c in cat_cols if c in df_info.columns]

df_info = create_categorical_encoding(df_info, cat_cols, method='label')
print(f"   ✅ {len(cat_cols)} variables categóricas codificadas")

# Edad: normalizar
if 'edad' in df_info.columns:
    df_info['edad'] = pd.to_numeric(df_info['edad'], errors='coerce')
    df_info['edad_normalizada'] = (df_info['edad'] - df_info['edad'].mean()) / df_info['edad'].std()
    print("   ✅ Edad normalizada")

# Aggregation features (por zona-fecha)
if 'fecha_hecho' in df_info.columns and 'barrios_hecho' in df_info.columns:
    df_info = create_aggregation_features(
        df_info, 
        group_cols=['barrios_hecho', 'nom_com'], 
        window_days=7
    )
    print("   ✅ Features de agregación creadas")

datasets_procesados['info_delictiva_bucaramanga'] = df_info
print(f"   📏 Shape final: {df_info.shape}")
print(f"   📋 Columnas: {len(df_info.columns)}")

# ============================================================================
# 3. DELITOS_SEXUALES
# ============================================================================
print("\n📊 3. Procesando: delitos_sexuales")
print("-"*80)

df_sex = dataframes['delitos_sexuales'].copy()

# Temporal features
if 'fecha_hecho' in df_sex.columns:
    df_sex = create_temporal_features(df_sex, date_col='fecha_hecho')
    print("   ✅ Features temporales creadas")

# Categorical encoding
cat_cols = ['departamento', 'municipio', 'armas_medios', 'genero', 'grupo_etario', 'delito']
cat_cols = [c for c in cat_cols if c in df_sex.columns]

df_sex = create_categorical_encoding(df_sex, cat_cols, method='label')
print(f"   ✅ {len(cat_cols)} variables categóricas codificadas")

# Aggregation features (por municipio)
if 'municipio' in df_sex.columns:
    df_sex = create_aggregation_features(df_sex, ['municipio', 'delito'])
    print("   ✅ Features de agregación creadas")

datasets_procesados['delitos_sexuales'] = df_sex
print(f"   📏 Shape final: {df_sex.shape}")
print(f"   📋 Columnas: {len(df_sex.columns)}")

# ============================================================================
# 4. VIOLENCIA_INTRAFAMILIAR
# ============================================================================
print("\n📊 4. Procesando: violencia_intrafamiliar")
print("-"*80)

df_viol = dataframes['violencia_intrafamiliar'].copy()

# Temporal features
if 'fecha_hecho' in df_viol.columns:
    df_viol = create_temporal_features(df_viol, date_col='fecha_hecho')
    print("   ✅ Features temporales creadas")

# Categorical encoding
cat_cols = ['departamento', 'municipio', 'armas_medios', 'genero', 'grupo_etario']
cat_cols = [c for c in cat_cols if c in df_viol.columns]

df_viol = create_categorical_encoding(df_viol, cat_cols, method='label')
print(f"   ✅ {len(cat_cols)} variables categóricas codificadas")

# Aggregation features
if 'municipio' in df_viol.columns:
    df_viol = create_aggregation_features(df_viol, ['municipio'])
    print("   ✅ Features de agregación creadas")

datasets_procesados['violencia_intrafamiliar'] = df_viol
print(f"   📏 Shape final: {df_viol.shape}")
print(f"   📋 Columnas: {len(df_viol.columns)}")

# ============================================================================
# 5. HURTO_MODALIDADES
# ============================================================================
print("\n📊 5. Procesando: hurto_modalidades")
print("-"*80)

df_hurto = dataframes['hurto_modalidades'].copy()

# Temporal features
if 'fecha_hecho' in df_hurto.columns:
    df_hurto = create_temporal_features(df_hurto, date_col='fecha_hecho')
    print("   ✅ Features temporales creadas")

# Categorical encoding
cat_cols = ['departamento', 'municipio', 'armas_medios', 'genero', 'grupo_etario', 'tipo_de_hurto']
cat_cols = [c for c in cat_cols if c in df_hurto.columns]

df_hurto = create_categorical_encoding(df_hurto, cat_cols, method='label')
print(f"   ✅ {len(cat_cols)} variables categóricas codificadas")

# Aggregation features
if 'municipio' in df_hurto.columns and 'tipo_de_hurto' in df_hurto.columns:
    df_hurto = create_aggregation_features(df_hurto, ['municipio', 'tipo_de_hurto'])
    print("   ✅ Features de agregación creadas")

datasets_procesados['hurto_modalidades'] = df_hurto
print(f"   📏 Shape final: {df_hurto.shape}")
print(f"   📋 Columnas: {len(df_hurto.columns)}")

print(f"\n{'='*80}")
print("✅ FEATURE ENGINEERING COMPLETADO")
print(f"{'='*80}")

# Resumen
print("\n📊 RESUMEN DE DATASETS PROCESADOS:\n")
summary_data = []
for key, df in datasets_procesados.items():
    summary_data.append({
        'Dataset': key,
        'Registros': f"{len(df):,}",
        'Columnas_Original': len(dataframes[key].columns),
        'Columnas_Procesado': len(df.columns),
        'Features_Nuevas': len(df.columns) - len(dataframes[key].columns),
        'Memoria_MB': f"{df.memory_usage(deep=True).sum() / 1024**2:.2f}"
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

FEATURE ENGINEERING POR DATASET

📊 1. Procesando: delitos_bucaramanga
--------------------------------------------------------------------------------
   ✅ Coordenadas normalizadas (ObjectToFloatTransformer)
   ✅ Coordenadas normalizadas (ObjectToFloatTransformer)
   ✅ Features espaciales creadas
   ✅ Features espaciales creadas
   ✅ 12 variables categóricas codificadas
   ✅ Features de agregación creadas
   📏 Shape final: (135076, 37)
   📋 Columnas: 37

📊 2. Procesando: info_delictiva_bucaramanga
--------------------------------------------------------------------------------
   ✅ 12 variables categóricas codificadas
   ✅ Features de agregación creadas
   📏 Shape final: (135076, 37)
   📋 Columnas: 37

📊 2. Procesando: info_delictiva_bucaramanga
--------------------------------------------------------------------------------
   ✅ Edad normalizada (ObjectToFloatTransformer)
   ✅ Features temporales creadas
   ✅ Edad normalizada (ObjectToFloatTransformer)
   ✅ Features temporales creadas

,Dataset,Registros,Columnas_Original,Columnas_Procesado,Features_Nuevas,Memoria_MB
0,delitos_bucaramanga,"135,076",19,37,18,141.48
1,info_delictiva_bucaramanga,"120,723",26,55,29,158.53
2,delitos_sexuales,"19,978",9,30,21,13.12
3,violencia_intrafamiliar,"50,864",8,27,19,24.84
4,hurto_modalidades,"1,422",9,30,21,0.81


## 4. Exportar Datasets Procesados

Guardamos los DataFrames procesados en formato **Parquet** que preserva:
- ✅ Nombres de columnas
- ✅ Tipos de datos (datetime, int, float, category)
- ✅ Compresión eficiente (~50% menor que CSV)
- ✅ Lectura rápida (10x más rápido que CSV)
- ✅ Compatible con pandas, dask, spark, polars

In [9]:
from datetime import datetime

# Crear carpeta de datasets procesados
processed_folder = Path('datasets/processed')
processed_folder.mkdir(parents=True, exist_ok=True)

print("="*80)
print("EXPORTANDO DATASETS PROCESADOS A PARQUET")
print("="*80)

saved_files = []

for dataset_name, df in datasets_procesados.items():
    print(f"\n📁 Guardando: {dataset_name}")
    
    # Ruta del archivo Parquet
    parquet_file = processed_folder / f"{dataset_name}_processed.parquet"
    
    # Guardar a Parquet con compresión
    df.to_parquet(
        parquet_file,
        engine='pyarrow',  # Motor más rápido y eficiente
        compression='snappy',  # Compresión rápida y eficiente
        index=False
    )
    
    # Calcular tamaño
    file_size_mb = parquet_file.stat().st_size / 1024 / 1024
    
    print(f"   ✅ Archivo: {parquet_file.name}")
    print(f"   📊 Shape: {df.shape}")
    print(f"   💾 Tamaño: {file_size_mb:.2f} MB")
    
    saved_files.append({
        'dataset': dataset_name,
        'archivo': parquet_file.name,
        'registros': len(df),
        'columnas': len(df.columns),
        'tamaño_mb': round(file_size_mb, 2)
    })

# Guardar metadata sobre el procesamiento
metadata = {
    'fecha_procesamiento': datetime.now().isoformat(),
    'formato': 'parquet',
    'compresion': 'snappy',
    'datasets': []
}

for dataset_name, df in datasets_procesados.items():
    # Identificar columnas nuevas (features creadas)
    cols_originales = set(dataframes[dataset_name].columns)
    cols_procesadas = set(df.columns)
    features_nuevas = list(cols_procesadas - cols_originales)
    
    metadata['datasets'].append({
        'nombre': dataset_name,
        'archivo': f"{dataset_name}_processed.parquet",
        'shape_original': list(dataframes[dataset_name].shape),
        'shape_procesado': list(df.shape),
        'columnas_originales': list(dataframes[dataset_name].columns),
        'columnas_procesadas': list(df.columns),
        'features_nuevas': features_nuevas,
        'tipos_datos': df.dtypes.astype(str).to_dict()
    })

# Guardar metadata
metadata_file = processed_folder / 'processing_metadata.json'
with open(metadata_file, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"\n{'='*80}")
print("RESUMEN DE ARCHIVOS GUARDADOS")
print(f"{'='*80}\n")

summary_df = pd.DataFrame(saved_files)
display(summary_df)

print(f"\n📦 Total de datasets guardados: {len(saved_files)}")
print(f"💾 Tamaño total: {sum(f['tamaño_mb'] for f in saved_files):.2f} MB")
print(f"📂 Ubicación: {processed_folder.absolute()}")
print(f"📋 Metadata: {metadata_file.name}")

print(f"\n✅ EXPORTACIÓN COMPLETADA")
print("="*80)

# Verificar archivos
print(f"\n📂 Archivos en '{processed_folder}':")
for file in sorted(processed_folder.glob('*.parquet')):
    size_mb = file.stat().st_size / 1024 / 1024
    print(f"   • {file.name} ({size_mb:.2f} MB)")

EXPORTANDO DATASETS PROCESADOS A PARQUET

📁 Guardando: delitos_bucaramanga
   ✅ Archivo: delitos_bucaramanga_processed.parquet
   📊 Shape: (135076, 37)
   💾 Tamaño: 4.60 MB

📁 Guardando: info_delictiva_bucaramanga
   ✅ Archivo: delitos_bucaramanga_processed.parquet
   📊 Shape: (135076, 37)
   💾 Tamaño: 4.60 MB

📁 Guardando: info_delictiva_bucaramanga
   ✅ Archivo: info_delictiva_bucaramanga_processed.parquet
   📊 Shape: (120723, 55)
   💾 Tamaño: 2.13 MB

📁 Guardando: delitos_sexuales
   ✅ Archivo: delitos_sexuales_processed.parquet
   📊 Shape: (19978, 30)
   💾 Tamaño: 0.22 MB

📁 Guardando: violencia_intrafamiliar
   ✅ Archivo: violencia_intrafamiliar_processed.parquet
   📊 Shape: (50864, 27)
   💾 Tamaño: 0.46 MB

📁 Guardando: hurto_modalidades
   ✅ Archivo: hurto_modalidades_processed.parquet
   📊 Shape: (1422, 30)
   💾 Tamaño: 0.04 MB

RESUMEN DE ARCHIVOS GUARDADOS

   ✅ Archivo: info_delictiva_bucaramanga_processed.parquet
   📊 Shape: (120723, 55)
   💾 Tamaño: 2.13 MB

📁 Guardando: d

,dataset,archivo,registros,columnas,tamaño_mb
0,delitos_bucaramanga,delitos_bucaramanga_processed.parquet,135076,37,4.60
1,info_delictiva_bucaramanga,info_delictiva_bucaramanga_processed.parquet,120723,55,2.13
2,delitos_sexuales,delitos_sexuales_processed.parquet,19978,30,0.22
3,violencia_intrafamiliar,violencia_intrafamiliar_processed.parquet,50864,27,0.46
4,hurto_modalidades,hurto_modalidades_processed.parquet,1422,30,0.04



📦 Total de datasets guardados: 5
💾 Tamaño total: 7.45 MB
📂 Ubicación: /home/juan/Solucion-Inteligente-de-Seguridad-Ciudadana-para-Santander/datasets/processed
📋 Metadata: processing_metadata.json

✅ EXPORTACIÓN COMPLETADA

📂 Archivos en 'datasets/processed':
   • delitos_bucaramanga_processed.parquet (4.60 MB)
   • delitos_sexuales_processed.parquet (0.22 MB)
   • hurto_modalidades_processed.parquet (0.04 MB)
   • info_delictiva_bucaramanga_processed.parquet (2.13 MB)
   • violencia_intrafamiliar_processed.parquet (0.46 MB)


---

## ✅ RESUMEN DE TRANSFORMACIONES APLICADAS

### 📊 **delitos_bucaramanga**
- ✅ Features espaciales: 6 nuevas (lat_grid, lon_grid, zona_id, dist_centro, count_by_zona_id, count_by_barrios_hecho)
- ✅ Categorical encoding: 13 variables → 13 columnas *_encoded
- ✅ **Total:** +19 columnas nuevas

### 📊 **info_delictiva_bucaramanga** (PRINCIPAL)
- ✅ Features temporales: 12 nuevas (año, mes, dia, dia_semana, fin_de_semana, trimestre, semestre, dia_mes, semana_año, nombre_dia, nombre_mes)
- ✅ Features espaciales (barrios): 1 nueva (barrio_delitos_count)
- ✅ Categorical encoding: 14 variables → 14 columnas *_encoded
- ✅ Edad normalizada: 1 nueva (edad_normalizada)
- ✅ Aggregations: 4 nuevas (count_by_barrios_hecho, count_by_nom_com, rolling_mean_7d, rolling_std_7d)
- ✅ **Total:** +32 columnas nuevas

### 📊 **delitos_sexuales, violencia_intrafamiliar, hurto_modalidades**
- ✅ Features temporales: 12 nuevas cada uno
- ✅ Categorical encoding: 5-6 variables por dataset
- ✅ Aggregations: por municipio y tipo de delito
- ✅ **Total:** +18-20 columnas nuevas por dataset

---

## 🎯 SIGUIENTES PASOS

1. **Ejecutar todas las celdas de este notebook** para generar los archivos `.parquet`
2. **Ir a `models.ipynb`** y ejecutar la celda de carga de datos
3. **Comenzar el entrenamiento** de modelos usando `datasets['info_delictiva_bucaramanga']` como principal

---